# Orthomosaic — Part 1: SfM Pipeline → Orthomosaic GeoTIFF

**Download drone images, extract metadata with `exif_gbx`, quality-control them with imagery scalars, and run distributed sparse SfM (pycolmap) to produce a georeferenced orthomosaic GeoTIFF.**

The pipeline combines GeoBrix data-source and scalar APIs with Databricks built-in geospatial functions:

- **GeoBrix `exif_gbx`** reads per-file EXIF + GPS metadata at scale (mode `"qc"` also computes sharpness and brightness without a second image decode).
- **GeoBrix `gsd_from_telemetry`** computes ground sampling distance from altitude, focal length, sensor size, and image width.
- **Databricks `ST_DistanceSphere`** filters image pairs by GPS distance — keeps pair count linear in image count instead of O(n²) exhaustive.

The `GROUP_KEY_COL` seam (config `"_group"`) enables multi-flight runs: set it to a metadata column (e.g. derived from `timestamp`) and the pipeline emits one orthomosaic per distinct value.

> **Runtime.** Runs on **Serverless environment 5** (lightweight tier). CPU-only sparse SfM; dense reconstruction is Phase 2.

---

**Last Update:** September 21, 2026

## Setup

In [ ]:
%run ./config_nb

## Stage the image dataset (idempotent)

Download the [Old Orchard DroneDB dataset](https://hub.dronedb.app/r/odm/old-orchard) from
GitHub. The `FORCE_DOWNLOAD=False` guard skips the fetch when images are already staged.

In [ ]:
import time as _time_import  # time imported in config_nb; alias for clarity

img_path = str(Path(DOWNLOAD_DIR).resolve())
staged   = list(Path(DOWNLOAD_DIR).rglob("*.jpg")) + list(Path(DOWNLOAD_DIR).rglob("*.jpeg"))

if FORCE_DOWNLOAD or not staged:
    _t = _time_import.perf_counter()
    files = list(iter_repo_files(GH_DATASET_PATH))
    image_files = [f for f in files if is_image(f["path"])]
    print(f"Fetching {len(image_files)} image files from GitHub...")
    for fi in image_files:
        download_file(fi, DOWNLOAD_DIR)
    staged = list(Path(DOWNLOAD_DIR).rglob("*.jpg")) + list(Path(DOWNLOAD_DIR).rglob("*.jpeg"))
    print(f"Download complete: {len(staged)} images staged in {_time_import.perf_counter()-_t:.1f}s")
else:
    print(f"{len(staged)} images already staged at {img_path} (FORCE_DOWNLOAD=False — skip)")

## Extract EXIF + GPS metadata with GeoBrix `exif_gbx`

**GeoBrix `exif_gbx`** — a Serverless-safe DataSource V2 reader — extracts per-file EXIF
metadata, GPS coordinates (as WKB geometry), focal length, sensor width, and image
dimensions without hand-rolling PIL loops.

`mode="qc"` additionally computes `sharpness` and `brightness` pixel metrics in the same
pass (no second image decode), saving one full round-trip over the file set.

In [ ]:
import time as _t_exif

_t0 = _t_exif.perf_counter()
try:
    # Build exif_gbx option overrides from config (only passed when not None).
    _exif_opts = {"mode": "qc"}
    if SENSOR_WIDTH_MM is not None:
        _exif_opts["sensorWidthMm"] = str(SENSOR_WIDTH_MM)
    if FOCAL_LENGTH_MM is not None:
        _exif_opts["focalLengthMm"] = str(FOCAL_LENGTH_MM)

    # spark.read.format("exif_gbx") — one row per image; no PIL loop needed.
    df_meta = (
        spark.read.format("exif_gbx")
             .options(**_exif_opts)
             .load(img_path)
    )

    # Add group key: default constant group so the pipeline runs as one batch.
    # Change GROUP_KEY_COL and assign a real column here for multi-group runs.
    if GROUP_KEY_COL == "_group":
        df_meta = df_meta.withColumn("_group", F.lit("all"))
    # else: GROUP_KEY_COL already exists in the schema (e.g. derived from timestamp)

    print(f"exif_gbx: {df_meta.count()} images read in {_t_exif.perf_counter()-_t0:.1f}s")
    df_meta.select(
        "path", "latitude", "longitude", "altitude",
        "focal_length_mm", "sensor_width_mm", "image_width", "image_height",
        "sharpness", "brightness",
    ).show(5, truncate=60)
except Exception as e:
    print(f"[ERROR] exif_gbx step failed after {_t_exif.perf_counter()-_t0:.1f}s: {e}")
    raise

## Quality-control filter

Remove blurry or underexposed images before SfM.

- **Sharpness** — keep images with sharpness ≥ 50% of the batch median.
- **Brightness** — keep images in the range [20, 240] (discard over- and under-exposed).

The `sharpness` and `brightness` columns were populated by `exif_gbx` mode `"qc"` using the
same `image_sharpness` / `image_brightness` scalars from `databricks.labs.gbx.pyrx.imagery`.

In [ ]:
import time as _t_qc

_t0 = _t_qc.perf_counter()
try:
    window_spec = Window.partitionBy(GROUP_KEY_COL)
    df_qc = (
        df_meta
        .withColumn("median_sharpness", F.percentile_approx("sharpness", 0.5).over(window_spec))
        .filter(F.col("sharpness") >= F.col("median_sharpness") * 0.5)
        .filter((F.col("brightness") > 20) & (F.col("brightness") < 240))
        .withColumn("gps_geom", F.expr("st_point(longitude, latitude)"))
        .withColumnRenamed("path", "source")
    )
    n_before = df_meta.count()
    n_after  = df_qc.count()
    print(f"QC filter: {n_before} → {n_after} images retained "
          f"({n_before - n_after} removed) in {_t_qc.perf_counter()-_t0:.1f}s")
except Exception as e:
    print(f"[ERROR] QC filter failed after {_t_qc.perf_counter()-_t0:.1f}s: {e}")
    raise

## Validate GPS coverage

Warn before SfM if fewer than 3 images have valid GPS — incremental mapping will proceed
but without GPS-constrained bundle adjustment, which reduces georeferencing accuracy.

In [ ]:
gps_ok_count = df_qc.filter(
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull() & F.col("altitude").isNotNull()
).count()
total_count = df_qc.count()
print(f"GPS coverage: {gps_ok_count}/{total_count} images have valid lat/lon/altitude")
if gps_ok_count < 3:
    print("[WARN] Fewer than 3 images have GPS — incremental mapping will proceed without "
          "GPS-constrained BA. Georeferencing accuracy may be reduced.")

## Run distributed SfM (one orthomosaic per `GROUP_KEY_COL` value)

For each distinct value of `GROUP_KEY_COL`, collect the image subset and run
`run_spark_sfm` — feature extraction, geospatial pair discovery, distributed matching,
and incremental mapping via `pycolmap`.

The `GROUP_KEY_COL = "_group"` default runs all images as a single group.
Set `GROUP_KEY_COL` to a metadata column (e.g. `"flight_date"`) and pass a real column
value to run the pipeline for each flight.

In [ ]:
import time as _t_sfm
from databricks.labs.gbx.pyrx.imagery import cluster_by_gps

# Per group: partition images into RAM-bounded GPS clusters, then run SfM per
# cluster. Peak driver RAM = one cluster reconstruction, not the whole scene.
# Feature extraction can hit a transient Serverless worker OOM (concurrent SIFT
# on a node); retry the cluster a few times before giving up (skip as last resort).
cluster_models = {}   # group -> {cluster_id: (sparse_dir, master_db)}
groups = [row[GROUP_KEY_COL] for row in df_qc.select(GROUP_KEY_COL).distinct().collect()]
print(f"Running clustered SfM for {len(groups)} group(s): {groups}")

_MAX_TRIES = 3
for grp in groups:
    _t0 = _t_sfm.perf_counter()
    print(f"\n=== Group: {grp} ===")
    df_grp = df_qc.filter(F.col(GROUP_KEY_COL) == grp)
    # cluster_by_gps runs on the small per-image telemetry table (driver-side).
    _pdf = df_grp.select("source", "latitude", "longitude").toPandas()
    _clustered = cluster_by_gps(
        _pdf,
        target_cluster_images=TARGET_CLUSTER_IMAGES,
        overlap_frac=CLUSTER_OVERLAP_FRAC,
        min_cluster_images=MIN_CLUSTER_IMAGES,
    )
    _cids = sorted(_clustered["_cluster"].unique())
    print(f"  {len(_pdf)} images \u2192 {len(_cids)} GPS cluster(s) "
          f"(target\u2248{TARGET_CLUSTER_IMAGES}, overlap={CLUSTER_OVERLAP_FRAC})")
    cluster_models[grp] = {}
    for cid in _cids:
        _srcs = _clustered.loc[_clustered["_cluster"] == cid, "source"].tolist()
        df_cluster = df_grp.filter(F.col("source").isin(_srcs))
        _co = f'{group_paths(grp)["sfm_dir"]}/cluster_{cid}'  # local SfM scratch (SQLite-safe)
        for _try in range(1, _MAX_TRIES + 1):
            _ct0 = _t_sfm.perf_counter()
            try:
                res = run_spark_sfm(
                    df_qc=df_cluster,
                    img_dir=img_path,
                    output_dir=_co,
                    group_key=f"{grp}_c{cid}",
                    do_overwrite_features=FORCE_RELOAD or _try > 1,
                    do_overwrite_matches=FORCE_RELOAD or _try > 1,
                    do_overwrite_master=FORCE_RELOAD or _try > 1,
                )
                cluster_models[grp][cid] = (res["sparse_dir"], res["master_db"])
                print(f"  cluster {cid}: {len(_srcs)} imgs, {res.get('registered')} registered "
                      f"in {_t_sfm.perf_counter()-_ct0:.0f}s")
                break
            except Exception as e:
                _m = str(e)
                _transient = ("OOM" in _m) or ("out of memory" in _m.lower()) or ("worker exited" in _m)
                if _transient and _try < _MAX_TRIES:
                    print(f"  [retry {_try}/{_MAX_TRIES-1}] cluster {cid} transient worker OOM "
                          f"after {_t_sfm.perf_counter()-_ct0:.0f}s \u2014 backoff + re-extract")
                    _t_sfm.sleep(15)
                    continue
                print(f"  [WARN] cluster {cid} ({len(_srcs)} imgs) failed after {_try} attempt(s): "
                      f"{_m[:180]} \u2014 skipped")
                break
    print(f"Group {grp!r}: {len(cluster_models[grp])} cluster model(s) "
          f"in {_t_sfm.perf_counter()-_t0:.0f}s")

## Compute GSD via GeoBrix `gsd_from_telemetry`

**GeoBrix `gsd_from_telemetry`** computes ground sampling distance in cm/pixel from
altitude (m), focal length (mm), sensor width (mm), and image width (px) using the
standard pinhole-camera formula:

```
GSD (cm/px) = sensor_mm × alt_m × 100 / (focal_mm × width_px)
```

This matches the formula used by OpenDroneMap and COLMAP documentation.

In [ ]:
import time as _t_gsd

_t0 = _t_gsd.perf_counter()
try:
    # Use exif_gbx columns (sensor_width_mm, focal_length_mm) + config override fallbacks.
    _sw_col = F.coalesce(F.col("sensor_width_mm"), F.lit(SENSOR_WIDTH_MM))
    _fl_col = F.col("focal_length_mm") if FOCAL_LENGTH_MM is None else F.lit(FOCAL_LENGTH_MM)

    df_gsd = df_qc.withColumn(
        "gsd_cm",
        gsd_udf(
            F.col("altitude").cast("double"),
            _fl_col.cast("double"),
            _sw_col.cast("double"),
            F.col("image_width").cast("double"),
        ),
    )
    stats   = df_gsd.agg(F.avg("gsd_cm"), F.min("gsd_cm"), F.max("gsd_cm")).collect()[0]
    avg_gsd = stats[0]
    print(f"GSD: avg={avg_gsd:.2f} cm/px  min={stats[1]:.2f}  max={stats[2]:.2f}  "
          f"({_t_gsd.perf_counter()-_t0:.1f}s)")

    # Use config GSD_CM if set; otherwise auto-derive from image telemetry.
    gsd_for_ortho = GSD_CM if GSD_CM is not None else round(avg_gsd, 1)
    print(f"Using GSD_CM={gsd_for_ortho:.1f} cm/px for orthomosaic rendering.")
except Exception as e:
    print(f"[ERROR] GSD step failed after {_t_gsd.perf_counter()-_t0:.1f}s: {e}")
    raise

## Build the orthomosaic GeoTIFF

Convert the COLMAP sparse reconstruction into a georeferenced RGB GeoTIFF via
`accumulate_orthomosaic`.  The canvas-RAM guard warns before allocating the float16 canvas
if the estimated memory exceeds ~4 GB (50% of typical Serverless 8 GB driver RAM).

In [ ]:
import time as _t_ortho

_t0 = _t_ortho.perf_counter()
try:
    # One blended orthomosaic per group: the group's GPS clusters are aligned to a
    # shared ENU frame and accumulated into a single canvas, written under group_<grp>/.
    ortho_paths = {}
    for grp in groups:
        cmods = cluster_models.get(grp, {})
        if not cmods:
            print(f"[WARN] group {grp!r}: no cluster models \u2014 skipping ortho")
            continue
        _gp = group_paths(grp)
        Path(_gp["ortho"]).parent.mkdir(parents=True, exist_ok=True)
        accumulate_orthomosaic(
            cluster_models=cmods,
            img_dir=img_path,
            output_tiff=_gp["ortho"],
            gsd_cm=gsd_for_ortho,
            max_workers=MAX_ORTHO_WORKERS,
            blend_gamma=BLEND_GAMMA,
        )
        ortho_paths[grp] = _gp["ortho"]
    print(f"Orthomosaic(s) built in {_t_ortho.perf_counter()-_t0:.0f}s \u2192 {list(ortho_paths.values())}")
except Exception as e:
    print(f"[ERROR] Orthomosaic build failed after {_t_ortho.perf_counter()-_t0:.1f}s: {e}")
    raise

## Visualise

In [ ]:
from pathlib import Path as _P
_shown = [g for g in groups if _P(group_paths(g)["ortho"]).exists()]
if _shown:
    show_raster(group_paths(_shown[0])["ortho"])
else:
    print("[skip] no orthomosaic produced this run (no cluster reconstructed)")

## Steps performed

1. **Download** — fetched the Old Orchard drone imagery from GitHub (idempotent).
2. **EXIF/GPS extraction** — `exif_gbx` (GeoBrix) read metadata + sharpness/brightness in one pass.
3. **QC filter** — retained images above the per-group sharpness p50 × 0.5 and within the brightness window.
4. **GPS validation** — warned when fewer than 3 images have valid altitude.
5. **Distributed SfM** — feature extraction, `ST_DistanceSphere` (Databricks) pair discovery, distributed matching, and incremental mapping via `pycolmap`.
6. **GSD** — computed via `gsd_from_telemetry` (GeoBrix) from altitude, focal length, sensor size, and image width.
7. **Orthomosaic** — back-projected all registered images onto the COLMAP ground plane; wrote an EPSG:4326 GeoTIFF.

**Next:** Part 2 applies per-channel percentile colour correction to remove sensor cast.